# OpenPlaque — Vendor Q3D Proximal Origin Audit v1.1 adjudication

This is the corrected adjudication of the same vendor-Q3D experiment.

The first run forced all radial views onto one global vertical axis. This version:
- measures every view along its own detected vessel axis;
- treats low-signal end views as uninformative rather than negative;
- uses repeated views i and i+12 as RCA radial-angle reproducibility pairs;
- allows only reproducibly oriented RCA angle pairs to determine the corresponding LAD/CX proximal end.

The original run remains preserved. This notebook writes to a new v1_1 output folder.

The 24 files remain radial curved views, not a physical 3-D stack. No clinical LM/LCX/OM identity is promoted and the frozen master is not modified.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Paths and reuse controls — immediately after Drive mount
DRIVE_ROOT = "/content/drive/MyDrive/OpenPlaque"
DICOM_ROOT = "/content/drive/MyDrive/CCTA/DICOM/3221"
OUT = f"{DRIVE_ROOT}/Vendor_Q3D_Proximal_Origin_Audit_v1_1"

print("DICOM root:", DICOM_ROOT)
print("Output:", OUT)
print("RCA Q3D:", f"{DICOM_ROOT}/32218")
print("CX Q3D:", f"{DICOM_ROOT}/32219")
print("LAD Q3D:", f"{DICOM_ROOT}/32220")


In [ ]:
import shutil, sys, subprocess, json
from pathlib import Path

OPENPLAQUE_PIN = "c7c4c80b9e08256219f0df37bf6809df75561b0f"
OPENPLAQUE_BRANCH = "vendor-q3d-proximal-origin-audit-from-main"

if Path("/content/OpenPlaque").exists():
    shutil.rmtree("/content/OpenPlaque")

!git clone -q --branch {OPENPLAQUE_BRANCH} https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git -C /content/OpenPlaque checkout -q {OPENPLAQUE_PIN}

%pip install -q pylibjpeg pylibjpeg-libjpeg
%pip install -q /content/OpenPlaque

for name in list(sys.modules):
    if name == "openplaque" or name.startswith("openplaque."):
        del sys.modules[name]

print("OpenPlaque pin:", subprocess.check_output(["git","-C","/content/OpenPlaque","rev-parse","HEAD"], text=True).strip())


In [ ]:
# Synthetic tests before study data
from openplaque.vendor_q3d_proximal_origin_audit_v1 import synthetic_self_test
print(synthetic_self_test())
!cd /content/OpenPlaque && pytest -q tests/test_vendor_q3d_proximal_origin_audit_v1.py

In [ ]:
from openplaque.vendor_q3d_proximal_origin_audit_v1 import run

for folder in ["32218","32219","32220","32210"]:
    p = Path(DICOM_ROOT)/folder
    if not p.is_dir():
        raise FileNotFoundError(p)

summary = run(
    dicom_root=DICOM_ROOT,
    drive_root=DRIVE_ROOT,
    output_dir=OUT,
)

print(json.dumps(summary["decision"], indent=2))
print("\nStatus:", summary["status"])

In [ ]:
# Primary quantitative review
import pandas as pd
from IPython.display import display

sig = pd.read_csv(Path(OUT)/"q3d_origin_signatures.csv")
views = pd.read_csv(Path(OUT)/"q3d_view_metrics.csv")
template = pd.read_csv(Path(OUT)/"q3d_rca_angle_orientation_template.csv")
pairs = pd.read_csv(Path(OUT)/"q3d_paired_view_reproducibility.csv")

print("RCA-calibrated origin signatures:")
display(sig)

print("\nRCA repeated-angle orientation template:")
display(template)

print("\nPaired-view raw-pixel reproducibility:")
display(pairs.groupby("vessel", as_index=False).agg(
    median_pixel_correlation=("pixel_correlation","median"),
    min_pixel_correlation=("pixel_correlation","min"),
    max_pixel_correlation=("pixel_correlation","max"),
))

allowed = template[template.usable_for_orientation]
print("\nAllowed RCA angles:", allowed.angle_index.tolist())

for vessel in ["RCA","LAD","CX"]:
    g = views[views.vessel==vessel].copy()
    print(f"\n{vessel}: informative views on their own detected axes")
    display(
        g[g.view_informative].sort_values("origin_score_margin", ascending=False)[
            ["view_index","angle_index","repeat_index","analysis_axis",
             "dominant_end","origin_score_margin",
             "left_origin_score","right_origin_score"]
        ].head(16)
    )


In [ ]:
# DICOM geometry / provenance review
geom = pd.read_csv(Path(OUT)/"q3d_geometry_inventory.csv")
refs = pd.read_csv(Path(OUT)/"q3d_source_reference_mapping.csv")
tags = pd.read_csv(Path(OUT)/"q3d_tag_inventory.csv")
private = pd.read_csv(Path(OUT)/"q3d_private_tag_inventory.csv")

print("Standard spatial geometry by vessel:")
display(
    geom.groupby("vessel", as_index=False).agg(
        n_images=("file","count"),
        standard_plane_geometry_images=("has_standard_plane_geometry","sum"),
        unique_frame_uids=("frame_of_reference_uid","nunique"),
    )
)

print("Series-7 source-reference linkage:")
display(
    refs.groupby("vessel", as_index=False).agg(
        total_referenced_sops=("referenced_sop_count","sum"),
        matched_series7_sops=("matched_series7_sop_count","sum"),
    )
)

print("Private tags whose values vary across the 24 radial views:")
display(
    private[private.n_unique_values>1][
        ["vessel","path","tag","keyword","name","vr","n_unique_values","example_values"]
    ].head(80)
)

In [ ]:
# Automatic image QC
from IPython.display import Image, display, HTML

display(Image(filename=str(Path(OUT)/"01_q3d_proximal_width_profiles.png"), width=1000))
for vessel in ["RCA","LAD","CX"]:
    p = Path(OUT)/f"QC_{vessel}_q3d_informative_views.png"
    print(p.name)
    display(Image(filename=str(p), width=1200))

report = Path(OUT)/"OPENPLAQUE_VENDOR_Q3D_PROXIMAL_ORIGIN_AUDIT_V1_1_REPORT.html"
display(HTML(report.read_text()))


In [ ]:
# Verify deliverables
expected = [
    "run_state.json",
    "summary.json",
    "decision.json",
    "q3d_geometry_inventory.csv",
    "q3d_tag_inventory.csv",
    "q3d_private_tag_inventory.csv",
    "q3d_source_reference_mapping.csv",
    "q3d_view_metrics.csv",
    "q3d_paired_view_reproducibility.csv",
    "q3d_rca_angle_orientation_template.csv",
    "q3d_origin_signatures.csv",
    "01_q3d_proximal_width_profiles.png",
    "QC_RCA_q3d_informative_views.png",
    "QC_LAD_q3d_informative_views.png",
    "QC_CX_q3d_informative_views.png",
    "OPENPLAQUE_VENDOR_Q3D_PROXIMAL_ORIGIN_AUDIT_V1_1_REPORT.html",
    "OPENPLAQUE_VENDOR_Q3D_PROXIMAL_ORIGIN_AUDIT_V1_1_RESULTS.zip",
]
missing=[x for x in expected if not (Path(OUT)/x).exists()]
if missing:
    raise RuntimeError("Missing outputs: "+str(missing))

state=json.loads((Path(OUT)/"run_state.json").read_text())
if state.get("status")!="COMPLETE":
    raise RuntimeError("Run state not COMPLETE: "+str(state))

print("COMPLETE")
for x in expected:
    print(Path(OUT)/x)
